# Module 5: Compare Local Ollama and Hugging Face

This experiment runs the same small sentiment task set through two workflows:

- **Ollama**: a local generative model called through `http://localhost:11434/api/generate`.
- **Hugging Face**: a pretrained text-classification pipeline from the Hub.

The goal is basic inference and evaluation. Fine-tuning is intentionally out of scope.

## Before you run

For Ollama, install Ollama, start it, and run `ollama pull llama3.2:3b`. You can replace the model with a DeepSeek variant that fits your hardware, such as `deepseek-r1:7b`.

For Hugging Face, the first run downloads `distilbert-base-uncased-finetuned-sst-2-english`. In Colab, run the install cell first.

In [ ]:
%pip install -q pandas requests transformers torch

import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"
HF_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

print("Configuration loaded. Change OLLAMA_MODEL before running if needed.")

## 1. Load one shared task set

The labels are held out from the model prompts and used only for evaluation.

In [ ]:
data_path = Path("data/test_data.csv")
if not data_path.exists():
    data_path = Path("module5_ollama_huggingface/data/test_data.csv")

tasks = pd.read_csv(data_path)
tasks

## 2. Hugging Face inference

This baseline is a binary sentiment model. A low confidence score is treated as `neutral` for this small teaching comparison; that is a practical heuristic, not a true three-class model.

In [ ]:
from transformers import pipeline

hf_classifier = pipeline("sentiment-analysis", model=HF_MODEL)

def normalize_hf_result(result):
    confidence = float(result["score"])
    if confidence < 0.65:
        return "neutral"
    return "positive" if result["label"].upper() == "POSITIVE" else "negative"

hf_rows = []
for row in tasks.itertuples(index=False):
    started = time.perf_counter()
    raw = hf_classifier(row.text, truncation=True)[0]
    hf_rows.append({
        "id": row.id,
        "text": row.text,
        "expected": row.label,
        "prediction": normalize_hf_result(raw),
        "confidence": round(float(raw["score"]), 4),
        "latency_ms": round((time.perf_counter() - started) * 1000, 1),
    })

hf_results = pd.DataFrame(hf_rows)
hf_results

## 3. Ollama inference through the API

The prompt asks for one JSON object so the response can be evaluated without manually interpreting prose. The health check makes the notebook usable even when Ollama is not installed or running.

In [ ]:
def ollama_is_available():
    try:
        response = requests.get(OLLAMA_URL.replace("/api/generate", "/api/tags"), timeout=3)
        return response.ok
    except requests.RequestException:
        return False

def extract_json(text):
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response: {text!r}")
    return json.loads(match.group(0))

def classify_with_ollama(text):
    prompt = (
        "Classify the sentiment of the text as exactly one of positive, negative, or neutral. "
        "Return only valid JSON with this schema: {\"label\": \"positive|negative|neutral\"}.\n\n"
        f"Text: {text}"
    )
    response = requests.post(
        OLLAMA_URL,
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "format": "json"},
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    parsed = extract_json(payload["response"])
    label = str(parsed.get("label", "")).lower().strip()
    if label not in {"positive", "negative", "neutral"}:
        raise ValueError(f"Unexpected label: {label!r}")
    return label

ollama_rows = []
if ollama_is_available():
    for row in tasks.itertuples(index=False):
        started = time.perf_counter()
        error = None
        try:
            prediction = classify_with_ollama(row.text)
        except (requests.RequestException, ValueError, KeyError) as exc:
            prediction = None
            error = str(exc)
        ollama_rows.append({
            "id": row.id,
            "text": row.text,
            "expected": row.label,
            "prediction": prediction,
            "latency_ms": round((time.perf_counter() - started) * 1000, 1),
            "error": error,
        })
else:
    print("Ollama is unavailable. Start Ollama and pull a model to run this side.")

ollama_results = pd.DataFrame(ollama_rows)
ollama_results

## 4. Compare accuracy and latency

Accuracy is exact label agreement. Latency is measured per example on the current machine, so it is useful for local observation rather than a universal benchmark.

In [ ]:
def summarize(name, results):
    if results.empty:
        return {"workflow": name, "status": "unavailable", "accuracy": None, "avg_latency_ms": None}
    valid = results[results.prediction.notna()]
    return {
        "workflow": name,
        "status": "complete" if len(valid) == len(results) else "partial",
        "accuracy": round((valid.prediction == valid.expected).mean(), 3) if not valid.empty else None,
        "avg_latency_ms": round(valid.latency_ms.mean(), 1) if not valid.empty else None,
        "examples_scored": len(valid),
    }

comparison = pd.DataFrame([
    summarize("Hugging Face", hf_results),
    summarize("Ollama", ollama_results),
])
comparison

In [ ]:
print("Per-example predictions")
display(hf_results[["id", "expected", "prediction", "confidence", "latency_ms"]])
if not ollama_results.empty:
    display(ollama_results[["id", "expected", "prediction", "latency_ms", "error"]])

## 5. Reflection

Write down your observations after running the cells:

1. Which workflow had higher accuracy on this task set?
2. Which workflow had lower latency, and what hardware or network conditions affected it?
3. How often did Ollama return valid structured output without repair?
4. What would you change before trusting this comparison: a larger labeled set, repeated runs, a three-class Hugging Face model, or a different local model?
5. Why is it better to stabilize inference and evaluation before fine-tuning?

### Colab note

The Hugging Face workflow is a natural Colab experiment. A standard Colab runtime does not expose your local Ollama daemon, so treat the Ollama result as unavailable unless you intentionally configure a reachable endpoint.